# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook provides a guided, reproducible analysis workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library, which is designed for interoperable, schema-driven dataset exploration and processing.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

In this section, we'll use the `mlcroissant` library to load the Croissant schema and dataset metadata. This gives us a programmatic interface to dataset structure, field definitions, and available data record sets.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and print overview
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Dataset Description: {metadata.description}\n")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}\n")
print(f"Licensing: {getattr(metadata, 'license', 'N/A')}\n")
print(f"Publication Date: {getattr(metadata, 'datePublished', 'N/A')}\n")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}\n")
print(f"Personal Sensitive Information Fields: {getattr(metadata, 'personalSensitiveInformation', 'N/A')}\n")

## 2. Data Overview

Review the available record sets, and, for each, list fields and columns referenced by their `@id`. This provides insight into the logical structure of the dataset before data extraction.


In [ ]:
# List all record sets
print("Available record sets and their fields (by @id):\n")
record_set_objs = dataset.record_sets

for rs in record_set_objs:
    print(f"RecordSet: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '<N/A>')}")
    print(f"  Description: {getattr(rs, 'description', '<N/A>')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}")
        print(f"      Name: {getattr(field, 'name', '<N/A>')}")
        print(f"      DataType: {getattr(field, 'data_type', '<N/A>')}")
        # List columns for each field if present
        if getattr(field, 'columns', None):
            for col in field.columns:
                print(f"      Column @id: {col.id}")
                print(f"        Name: {getattr(col, 'name', '<N/A>')}")
        else:
            print("      Columns: <none>")
    print('')

## 3. Data Extraction

Now, we'll load data from a specific record set into a Pandas DataFrame for further analysis.

- First, collect all record set `@id`s.
- We'll use the first record set for demonstration, but you can adapt for others by changing the `record_set_id` variable.


In [ ]:
# Extract all record set IDs
record_sets = [rs.id for rs in dataset.record_sets]
print('Record Set @ids:', record_sets)

# For demonstration, select the first record set:
if not record_sets:
    raise ValueError('No record sets were found in the dataset.')
record_set_id = record_sets[0]

# Load data from the chosen record set
df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))

print(f"DataFrame columns (field @id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform typical analyses:
- Filter records based on a numeric field (e.g., age at CRC diagnosis)
- Normalize this variable
- Group and aggregate by a key categorical attribute (e.g., sex)

**Note:** Be sure to replace field `@id` values and thresholds with those specific to your dataset, found above in the 'Data Overview'.


In [ ]:
# Choose a numeric field (update this as needed—replace the @id with the correct one)
# Inspect which columns look numeric:
display(df.dtypes)

# Let's try with 'cr:field/age_at_crc_diagnosis' if available (update accordingly)
numeric_field = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    # fallback for demonstration
    numeric_field = df.columns[df.dtypes == float].tolist()[0] if (df.dtypes == float).any() else df.columns[0]
print(f'Using numeric field: {numeric_field}')

# Set a threshold (adjust for your data context)
threshold = 50  # e.g., filter patients older than 50

if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, col_norm]].head())

    # Group by a categorical field (e.g., sex or anatomical_site)
    group_field = None
    for col in df.columns:
        if 'sex' in col.lower() or 'gender' in col.lower() or 'site' in col.lower():
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df)
else:
    print(f'Field {numeric_field} not found in DataFrame columns.')

## 5. Visualization

Let's visualize the distribution of a numeric field (e.g., age at CRC diagnosis) and its relationship to a categorical field (e.g., sex or anatomical_site).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot of age by sex (update group_field as needed)
if numeric_field and group_field and group_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"Distribution of {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()
else:
    print('Appropriate fields for plotting were not found in the DataFrame.')

## 6. Conclusion

In this notebook, you:
- Explored dataset structure and record sets from a Croissant schema via `mlcroissant`.
- Loaded a record set into a Pandas DataFrame using only `@id` references.
- Performed typical cleaning and analysis steps, including numeric filtering, normalization, and grouping.
- Visualized the relationship between demographic and clinical variables.

You can extend this workflow by: analyzing additional record sets, exploring alternative variables, or building predictive/statistical models. Refer to the dataset's Croissant schema for further field definitions and data provenance.